# Practical Exam: Customer Purchase Prediction

RetailTech Solutions is a fast-growing international e-commerce platform operating in over 20 countries across Europe, North America, and Asia. They specialize in fashion, electronics, and home goods, with a unique business model that combines traditional retail with a marketplace for independent sellers.

The company has seen rapid growth. A key part of their success has been their data-driven approach to personalization. However, as they plan their expansion into new markets, they need to improve their ability to predict customer behavior.

Their marketing team wants to predict which customers are most likely to make a purchase based on their browsing behavior.

As an AI Engineer, you will help build this prediction system. Your work will directly impact RetailTech's growth strategy and their goal of increasing revenue.


## Data Description

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

# Task 1

The marketing team has collected customer session data in `raw_customer_data.csv`, but it contains missing values and inconsistencies that need to be addressed.
Create a cleaned version of the dataframe:

- Start with the data in the file `raw_customer_data.csv`
- Your output should be a DataFrame named `clean_data`
- All column names and values should match the table below.
</br>

| Column Name | Criteria |
|------------|----------|
| customer_id | Integer. Unique identifier for each customer. No missing values. |
| time_spent | Float. Minutes spent on website per session. Missing values should be replaced with median. |
| pages_viewed | Integer. Number of pages viewed in session. Missing values should be replaced with mean. |
| basket_value | Float. Value of items in basket. Missing values should be replaced with 0. |
| device_type | String. One of: Mobile, Desktop, Tablet. Missing values should be replaced with "Unknown". |
| customer_type | String. One of: New, Returning. Missing values should be replaced with "New". |
| purchase | Binary. Whether customer made a purchase (1) or not (0). Target variable. |

In [10]:
# Task 1: Clean and standardize data types; handle missing values appropriately
import pandas as pd

clean_data = pd.read_csv('raw_customer_data.csv')

# customer_id: integer, no missing values
clean_data['customer_id'] = clean_data['customer_id'].astype(int)

# time_spent: float, missing -> median
clean_data['time_spent'] = clean_data['time_spent'].astype(float)
clean_data['time_spent'] = clean_data['time_spent'].fillna(clean_data['time_spent'].median())

# pages_viewed: integer, missing -> mean
clean_data['pages_viewed'] = clean_data['pages_viewed'].fillna(clean_data['pages_viewed'].mean())
clean_data['pages_viewed'] = clean_data['pages_viewed'].astype(int)

# basket_value: float, missing -> 0
clean_data['basket_value'] = clean_data['basket_value'].astype(float)
clean_data['basket_value'] = clean_data['basket_value'].fillna(0)

# device_type: string, missing -> "Unknown"
clean_data['device_type'] = clean_data['device_type'].fillna('Unknown')

# customer_type: string, missing -> "New"
clean_data['customer_type'] = clean_data['customer_type'].fillna('New')

# purchase: binary target, no missing values
clean_data['purchase'] = clean_data['purchase'].astype(int)

clean_data


,customer_id,time_spent,pages_viewed,basket_value,device_type,customer_type,purchase
0,1,23.097867,7,50.574647,Mobile,Returning,0
1,2,57.092144,3,56.891022,Mobile,Returning,1
2,3,44.187643,14,8.348296,Mobile,Returning,0
3,4,36.320851,10,43.481489,Mobile,New,1
4,5,10.205100,16,0.000000,Mobile,Returning,1
...,...,...,...,...,...,...,...
495,496,21.847781,6,39.954545,Mobile,New,1
496,497,35.435711,15,64.972694,Desktop,Returning,1
497,498,35.037329,10,0.000000,Unknown,New,1
498,499,58.489294,5,73.736271,Mobile,Returning,1


# Task 2
The pre-cleaned dataset `model_data.csv` needs to be prepared for our neural network.
Create the model features:

- Start with the data in the file `model_data.csv`
- Scale numerical features (`time_spent`, `pages_viewed`, `basket_value`) to 0-1 range
- Apply one-hot encoding to the categorical features (`device_type`, `customer_type`)
    - The column names should have the following format: variable_name_category_name (e.g., `device_type_Desktop`)
- Your output should be a DataFrame named `model_feature_set`, with all column names from `model_data.csv` except for the columns where one-hot encoding was applied.


In [11]:
# Task 2: Prepare data for modeling
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

model_data = pd.read_csv('model_data.csv')
model_feature_set = model_data.copy()

# Scale numerical features to 0-1 range
num_cols = ['time_spent', 'pages_viewed', 'basket_value']
scaler = MinMaxScaler()
model_feature_set[num_cols] = scaler.fit_transform(model_feature_set[num_cols])

# One-hot encode categorical features -> variable_name_category_name format
model_feature_set = pd.get_dummies(model_feature_set, columns=['device_type', 'customer_type'])

# Ensure one-hot columns are integer (0/1) rather than boolean
dummy_cols = [c for c in model_feature_set.columns
              if c.startswith('device_type_') or c.startswith('customer_type_')]
model_feature_set[dummy_cols] = model_feature_set[dummy_cols].astype(int)

model_feature_set


,customer_id,time_spent,pages_viewed,basket_value,purchase,device_type_Desktop,device_type_Mobile,device_type_Tablet,device_type_Unknown,customer_type_New,customer_type_Returning
0,501,0.664167,0.500000,0.000000,1,1,0,0,0,1,0
1,502,0.483681,0.222222,0.524981,1,0,1,0,0,0,1
2,503,0.231359,0.111111,0.457291,0,0,1,0,0,0,1
3,504,0.792944,0.277778,0.000000,1,0,0,0,1,1,0
4,505,0.649210,0.166667,0.484283,1,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
495,996,0.510473,1.000000,0.459799,1,1,0,0,0,1,0
496,997,0.908229,0.000000,0.000000,0,0,0,1,0,0,1
497,998,0.039019,0.333333,0.202147,1,1,0,0,0,0,1
498,999,0.944895,0.888889,0.369052,1,0,1,0,0,0,1


# Task 3

Now that all preparatory work has been done, create and train a neural network that would allow the company to predict purchases.

- Using PyTorch, create a network with:
   - At least one hidden layer with 8 units
   - ReLU activation for hidden layer
   - Sigmoid activation for the output layer
- Using the prepared features in `input_model_features.csv`, train the model to predict purchases. 
- Use the validation dataset `validation_features.csv` to predict new values based on the trained model. 
- Your model should be named `purchase_model` and your output should be a DataFrame named `validation_predictions` with columns `customer_id` and `purchase`. The `purchase` column must be your predicted values.


In [12]:
# Task 3: Implement standard modeling approaches - create, train, and validate network model
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

torch.manual_seed(42)

# ---- Load prepared training features ----
train_df = pd.read_csv('input_model_features.csv')
val_df = pd.read_csv('validation_features.csv')

feature_cols = [c for c in train_df.columns if c not in ('customer_id', 'purchase')]

X = train_df[feature_cols].values.astype(np.float32)
y = train_df['purchase'].values.astype(np.float32).reshape(-1, 1)

# Hold out a split from the training data to validate model performance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

# ---- Define the network ----
class PurchaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.hidden = nn.Linear(input_dim, 8)
        self.relu = nn.ReLU()
        self.output = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.hidden(x))
        x = self.sigmoid(self.output(x))
        return x

purchase_model = PurchaseNet(input_dim=X_train.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(purchase_model.parameters(), lr=0.01)

# ---- Train ----
epochs = 200
for epoch in range(epochs):
    purchase_model.train()
    optimizer.zero_grad()
    outputs = purchase_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

# ---- Validate performance on held-out split ----
purchase_model.eval()
with torch.no_grad():
    test_preds = (purchase_model(X_test_t) >= 0.5).float()
    accuracy = (test_preds.eq(y_test_t).sum() / y_test_t.shape[0]).item()
print(f"Held-out validation accuracy: {accuracy:.4f}")

# ---- Predict on the validation_features.csv dataset ----
X_val = val_df[feature_cols].values.astype(np.float32)
X_val_t = torch.tensor(X_val)

with torch.no_grad():
    val_probs = purchase_model(X_val_t)
    val_preds = (val_probs >= 0.5).int().numpy().flatten()

validation_predictions = pd.DataFrame({
    'customer_id': val_df['customer_id'],
    'purchase': val_preds
})

validation_predictions


Held-out validation accuracy: 0.7875


,customer_id,purchase
0,1801,1
1,1802,1
2,1803,1
3,1804,1
4,1805,1
...,...,...
195,1996,1
196,1997,1
197,1998,1
198,1999,1
